# Goal

This notebook reproduces the minor errors encountered when I tried to replicate the paper: "_Computational exploration of global venoms for antimicrobial discovery with Venomics artificial intelligence_". (Paper [link](https://www.nature.com/articles/s41467-025-60051-6) )




In [1]:
#@title  Let us begin with the basic imports
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from scipy.linalg import block_diag
# Don't do linear algebra in Python without these two lines
np.set_printoptions(suppress=True)
from collections import Counter
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%precision 3
#############################################
import sys
import importlib
importlib.reload(sys)
#######################
from google.colab import drive
drive.flush_and_unmount()
import os
drive.mount('/gdrive', force_remount=True)
# Enter your own proj_dir here
proj_dir='/gdrive/My Drive/venomics.ai/AI/VENOMICS_UPENN/'
os.chdir(proj_dir)

/tmp/ipykernel_35663/3030987161.py:14: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


Mounted at /gdrive


In [2]:
#@title  Install the requisite packages and make sure you have a decent GPU (T4 suffices)
!pip install -q fair-esm biopython rdkit
!nvidia-smi

Tue May 26 19:12:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Issue 1 - Minor : Non-VenomZone qualifier amiss / Missing VEPs

 Supplementary data was supposed to represent all the VEPs from across the 4 constituent datasets predicted by APEX. Source: [This](https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM2_ESM.pdf) supplemtary pdf.

```
Description of Additional supplementary files:

- **Supplementary data 1:** List of VEP predicted by APEX to have a median MIC ≤32 μmol<sup>-1</sup>.csv

- **Supplementary data 2:** List of VEP identified by APEX and filter criterion
```

But it appears that the data only captures the subset of VEPs from VenomZone (Of the 16123 protein sequences, 7769 were derived from VenomZone).

In [3]:
url_1='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM3_ESM.csv'

url_2='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM4_ESM.csv'

df_1=pd.read_csv(url_1)
df_2=pd.read_csv(url_2)

df_1.shape,df_2.shape

((4618, 13), (273, 16))

OK. So `df_1`has 4618 rows and df_2 has 273.

The only context in which 4618 and 273 appear is in the context of the "VenomZone(UniProtPK)" sub-dataset (Refer to the _Supplementary Table 1. Database-sourced venom protein and VEP candidates_ of the supplementary section found [here](https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM1_ESM.pdf)).

 Image of the table below:

 ![Supp Table 1](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/8d087f786b2151feb02947226f0285d75ad3789a/images/img_4618.png)


 https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM7_ESM.xlsx

# Issue -2:  Overlapping peptide sequences in the final 58

 I think there are only 55 unique peptide sequences amongst the 58 declared!!

In [4]:
url_58='https://github.com/vinayprabhu/APEX_reproduce/raw/refs/heads/main/df_final_58.tsv'
df_vep_58=pd.read_csv(url_58,sep='\t')
df_vep_58.Sequence.value_counts().head()

,count
Sequence,
KLLKIGLKSFARVLKKVL,2
KNKRFIRNLRSNLYQKIIKSTKSLL,2
KRRRASPLWKRRRFLSMLKARAK,2
LKLKSILGKLGVIL,1
RRVKRFKKFFMKLKKSVKKRVMKFFK,1


In [5]:
df_vep_58.Sequence.value_counts()[df_vep_58.Sequence.value_counts()>1].index

Index(['KLLKIGLKSFARVLKKVL', 'KNKRFIRNLRSNLYQKIIKSTKSLL',
       'KRRRASPLWKRRRFLSMLKARAK'],
      dtype='object', name='Sequence')

Yes indeed!

Confirmed that ['KLLKIGLKSFARVLKKVL', 'KNKRFIRNLRSNLYQKIIKSTKSLL',
       'KRRRASPLWKRRRFLSMLKARAK'] are repeating via actual text search
       

![3 are duplicates](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/7d8e4ab1868a629ca07332fa8a7f5f41f37c54ce/images/img_58.png
)



In [6]:
list_55=df_vep_58.Sequence.unique()
list_55

array(['KLLKIGLKSFARVLKKVL', 'WLGSALKIGAKLL', 'KLWNSKLARKIRTKGLKYVKNFAK',
       'LKLKSILGKLGVIL', 'RRVKRFKKFFMKLKKSVKKRVMKFFK', 'GKWLISSLVAKHL',
       'KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK', 'KLKKLRKWIYRIV',
       'FLKKIWRSKLVKRL', 'KRRRASPLWKRRRFLSMLKARAK', 'LTKWLGKLGVIL',
       'RKFKWGKLFSTAKKLYKKGKKLSKNKNFKKALK', 'KFLARLVFRKFILL',
       'KNKRFIRNLRSNLYQKIIKSTKSLL', 'KWLGKLGVILSHL',
       'RKFKWGKLFSTAKKLYKKGKKLSK', 'FIKKLWRSKLAKKLRAKGRELLK',
       'RRVKRFKKFFMKL', 'VNSFKIGGFIKKLWRSKLAKKLRAK', 'RFGSFLKKVWKSKLAKKL',
       'RRVKRFKKFFRKLKKSVKKRAKEFFK', 'RHRIVRTYIAKFGLK',
       'KRKGYLRLVPEERIWQKGLWWLRRLETDSDKLQK', 'LLHFSIWRSTVLRK',
       'RHRIVRTYIAKFGLKLNEFFQENENAWYFIRNIRKRVWEVKK', 'RRVKRFKKFFKKL',
       'QPRRVKRFKKFFKKLKNSVKKRAKKF', 'KRLRAKMLNSKFIKLIKR',
       'RRASPLWKRRRFLSMLKARAKRTGYK', 'PLWKRRRFLSMLKARAKR',
       'LRAKMLNSKFIKL', 'KLHGLLTRRSLKNFWKRNLYLR',
       'KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR', 'RLRAKMRNSKLFKLTKR',
       'RQEYPTKRLRAKMLNSKFIKLIKR', 'KKWRELSRLSRVLQIL'

# Issue 3 : The missing conoserver sequences!

![conoserver screenshot May 15 2026](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/a73523b4807a1ccecc76bc346fa416239c7d00e6/images/conoserver.png)

The main issue:

- The final VEPs attributed to Conoserver were not found in the fa file!

In [7]:
#@title  First, let us carve out the 12 conoserver VEPs that were predicted by APEX
df_cs=df_vep_58.loc[df_vep_58.Peptide.str.contains('Conoserver', na=False), :]
concoserver_12=df_cs.Sequence.unique()
print(concoserver_12)
df_cs

['KRLRAKMLNSKFIKLIKR' 'KRRRASPLWKRRRFLSMLKARAK'
 'RRASPLWKRRRFLSMLKARAKRTGYK' 'PLWKRRRFLSMLKARAKR' 'LRAKMLNSKFIKL'
 'KLHGLLTRRSLKNFWKRNLYLR' 'KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR'
 'RLRAKMRNSKLFKLTKR' 'RQEYPTKRLRAKMLNSKFIKLIKR' 'KKWRELSRLSRVLQIL'
 'RKRRRFISMLKARAKRR' 'KQKYLIKRSRAKMQNHKLFKLTKR']


,Peptide,Sequence
27,Conoserver-1,KRLRAKMLNSKFIKLIKR
28,Conoserver-2,KRRRASPLWKRRRFLSMLKARAK
29,Conoserver-3,RRASPLWKRRRFLSMLKARAKRTGYK
30,Conoserver-4,PLWKRRRFLSMLKARAKR
31,Conoserver-5,LRAKMLNSKFIKL
32,Conoserver-6,KLHGLLTRRSLKNFWKRNLYLR
33,Conoserver-7,KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR
34,Conoserver-10,RLRAKMRNSKLFKLTKR
35,Conoserver-12,RQEYPTKRLRAKMLNSKFIKLIKR
36,Conoserver-14,KKWRELSRLSRVLQIL


In [8]:
#@title Now, let us fetch all the conoserver data and extract the sequences!

def check_canonical_gt8(seq_c):
  """
  Function for filtering out the non-canonical sequences and sequences with length <8
  """
  CANONICAL = set("ACDEFGHIKLMNPQRSTVWY")
  iscanon=not (set(seq_c.upper()) - CANONICAL)
  isgt8=len(seq_c)>=8
  return iscanon & isgt8

from Bio import SeqIO
import requests
from io import StringIO

# URL to the raw FASTA file
url = "https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/main/conoserver_250206_protein.fa"

# Fetch the content and parse it using Biopython
response = requests.get(url)
fasta_data = StringIO(response.text)

# Define the expected columns
cols = [
    'conoserver_identifier', 'name', 'organism', 'protein type',
    'toxin class', 'gene_superfamily', 'cysteine_framework',
    'pharmacological_family', 'evidence'
]

# Extract header components and sequence
data = []
for record in SeqIO.parse(fasta_data, "fasta"):
    # Split the description line by '|'
    # Assuming the header follows the exact order of your column list
    header_parts = record.description.split('|')

    # Append the sequence to the list of header parts
    row = header_parts + [str(record.seq)]
    data.append(row)

# Create DataFrame
# Note: Added 'sequence' to columns since that is extracted from the FASTA
df_cs_raw = pd.DataFrame(data, columns=cols + ['sequence'])

# View the result
print(f'The current version has {df_cs_raw.shape[0]} seqs')

c_vec=df_cs_raw.sequence.apply(check_canonical_gt8)
# Apply the filter directly to the raw dataframe
df_cs_filt = df_cs_raw[c_vec].reset_index(drop=True)
# Verify the count
print(f"Filtered DataFrame shape: {df_cs_filt.shape}")

df_cs_filt.head(4)


The current version has 8523 seqs
Filtered DataFrame shape: (6603, 10)


,conoserver_identifier,name,organism,protein type,toxin class,gene_superfamily,cysteine_framework,pharmacological_family,evidence,sequence
0,P00005,PeIA precursor,Conus pergrandis,Precursor,conotoxin,A superfamily,,,,FDGRNAAANDKASDLVALTVRGCCSHPACSVNHPELCG
1,P00008,MII precursor,Conus magus,Precursor,conotoxin,A superfamily,,,,MGMRMMFTVFLLVVLATTVVSFPSDRASDGRNAAANDKASDVITLA...
2,P00009,SII precursor,Conus striatus,Precursor,conotoxin,A superfamily,,,,MGMRMMFTVFLLVVLATTVVSFPSDRASDGRDDEAKDERSDMHESD...
3,P00011,Ca1.1 precursor,Conus caracteristicus,Precursor,conotoxin,A superfamily,,,nucleic acid level,MGMRMMFTVFLLVVLATTVVSFTSDRASDGRNAAANAFDLIALIAR...


In [9]:
#@title Finally, let us see how many of the final 58 sequences in df_cs are actually in df_cs_filt
##############################

def find_substring_locations(df, query_string):
  # Helper function for sub-string search
    # .map() replaces the deprecated .applymap()
    mask = df.map(lambda x: query_string in str(x))

    # Filter for rows containing the substring
    results = df[mask.any(axis=1)]

    return results
def print_list_in_red(my_list):
    # ANSI escape code for red text and reset
    RED = "\033[91m"
    RESET = "\033[0m"

    for element in my_list:
        print(f"{RED}{element}{RESET}")

def print_highlighted_substring(full_string, substring):
  # Helper function for sub-string printing with highlighting
    if not substring:
        print(full_string)
        return

    # ANSI escape code for green text and reset
    GREEN = "\033[92m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    # Find all starting indices of the substring
    start = 0
    indices = []
    while True:
        idx = full_string.find(substring, start)
        if idx == -1:
            break
        indices.append(idx)
        start = idx + 1

    if not indices:
        print(f"Substring '{substring}' not found.")
        return

    # Build the highlighted string
    highlighted = ""
    last_idx = 0
    for idx in indices:
        highlighted += full_string[last_idx:idx]
        # Wrap substring in ANSI codes
        highlighted += f"{GREEN}{BOLD}{substring}{RESET}"
        last_idx = idx + len(substring)

    highlighted += full_string[last_idx:]
    print(highlighted)
#################################

list_conoserver_notfound=[]
for seq_ in df_cs.Sequence.values:
  results=find_substring_locations(df_cs_filt,seq_)
  print(seq_)
  if(results.shape[0]>0):
    print(results[['conoserver_identifier', 'name', 'organism']])
    print_highlighted_substring(results['sequence'].values[0], seq_)
  else:
    print(f'{seq_} not found')
    list_conoserver_notfound.append(seq_)
  print('-------------------------')

print('Not found in list_conoserver: ')
print_list_in_red(list_conoserver_notfound)

KRLRAKMLNSKFIKLIKR
     conoserver_identifier              name               organism
5189                P07699  Ca6.13 precursor  Conus caracteristicus
MKLTCALIVAMLLLTACQLTTADASRGRQEYPTKRLRAKMLNSKFIKLIKRCAAPGASCSKYDNECCDACLLQYPNPPVC
-------------------------
KRRRASPLWKRRRFLSMLKARAK
     conoserver_identifier                  name          organism
5051                P06892  Con-ins G2 precursor  Conus geographus
MTTSSYFLLVALGLLLYVRQSFSTHEHTCQLDDPAHPQGKCGSDLVNYHEEKCEEEEARRGGTNDGGKKRRRASPLWKRRRFLSMLKARAKRTGYKGIACECCQHYCTDQEFINYCPPVTESSSSSSSAA
-------------------------
RRASPLWKRRRFLSMLKARAKRTGYK
     conoserver_identifier                  name          organism
5051                P06892  Con-ins G2 precursor  Conus geographus
MTTSSYFLLVALGLLLYVRQSFSTHEHTCQLDDPAHPQGKCGSDLVNYHEEKCEEEEARRGGTNDGGKKRRRASPLWKRRRFLSMLKARAKRTGYKGIACECCQHYCTDQEFINYCPPVTESSSSSSSAA
-------------------------
PLWKRRRFLSMLKARAKR
     conoserver_identifier                  name          organism
5051               

# Issue 4:  Mismatch of MIC values via predict.py (Perhaps requires sub-indexing over 11 strains and not all 34)
Source: https://gitlab.com/machine-biology-group-public/apex/-/raw/main/predict.py


In [10]:
#@title Let us first load the 40 Apex model family
%cd apex
import os
import json
#from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math, copy, time
from torch.autograd import Variable
from scipy import stats
import pandas as pd
from sklearn.model_selection import KFold
import pickle
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import StepLR
import os.path
from Bio import SeqIO
import string
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from AMP_DL_model_twohead import AMP_model
#from propy.AAComposition import CalculateAADipeptideComposition
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy import stats
from utils import *
from scipy import sparse
import sys
from optparse import OptionParser
import copy
import pandas as pd


col = ['E. coli ATCC11775', 'P. aeruginosa PAO1', 'P. aeruginosa PA14', 'S. aureus ATCC12600', 'E. coli AIG221', 'E. coli AIG222', 'K. pneumoniae ATCC13883', 'A. baumannii ATCC19606', 'A. muciniphila ATCC BAA-835', 'B. fragilis ATCC25285', 'B. vulgatus ATCC8482', 'C. aerofaciens ATCC25986', 'C. scindens ATCC35704', 'B. thetaiotaomicron ATCC29148', 'B. thetaiotaomicron Complemmented', 'B. thetaiotaomicron Mutant', 'B. uniformis ATCC8492', 'B. eggerthi ATCC27754', 'C. spiroforme ATCC29900', 'P. distasonis ATCC8503', 'P. copri DSMZ18205', 'B. ovatus ATCC8483', 'E. rectale ATCC33656', 'C. symbiosum', 'R. obeum', 'R. torques', 'S. aureus (ATCC BAA-1556) - MRSA', 'vancomycin-resistant E. faecalis ATCC700802', 'vancomycin-resistant E. faecium ATCC700221', 'E. coli Nissle', 'Salmonella enterica ATCC 9150 (BEIRES NR-515)', 'Salmonella enterica (BEIRES NR-170)', 'Salmonella enterica ATCC 9150 (BEIRES NR-174)', 'L. monocytogenes ATCC 19111 (BEIRES NR-106)']

max_len = 52 # maximum peptide length

word2idx, idx2word = make_vocab()
emb, AAindex_dict = AAindex('aaindex1.csv', word2idx)
#Sourced from https://gitlab.com/machine-biology-group-public/apex/-/blob/main/aaindex1.csv?ref_type=heads
vocab_size = len(word2idx)
emb_size = np.shape(emb)[1]


model_num = 8
repeat_num = 5



f = open('best_key_list', 'r')
# Sourced from https://gitlab.com/machine-biology-group-public/apex/-/blob/main/best_key_list?ref_type=heads
lines = f.readlines()
f.close()

model_list = []
for line in lines:
  parsed = line.strip('\n').strip('\r')
  model_list.append(parsed)


all_list = []
ensemble_num = model_num * repeat_num

deep_model_list = []
for a_model_name in model_list:
  for a_en in range(repeat_num):
    key = 'trained_all_model_'+a_model_name+'_ensemble_'+str(a_en)

    #model = torch.load('./trained_models/'+key)
    model = torch.load('./trained_models/'+key, weights_only=False)
    model.eval()
    deep_model_list.append(model)


/gdrive/My Drive/venomics.ai/AI/VENOMICS_UPENN/apex


In [11]:
#@title Issue 4a : I think, the procedure for median thresholding entails indexing only on the 11 strains and not all the 34 from APEX's output
seq_list=df_vep_58.Sequence.unique()
print(f' The input list is: {seq_list}')

import time
t=time.time()


ensemble_counter = 0
for ensemble_id in range(ensemble_num):

	AMP_model = deep_model_list[ensemble_id].cuda().eval()

	data_len = len(seq_list)
	batch_size = 3000 #change according to your GPU memory
	for i in range(int(math.ceil(data_len/float(batch_size)))):
		# if (i*batch_size) % 1000 == 0:
		# 	print ('progress', i*batch_size, data_len)

		seq_batch = seq_list[i*batch_size:(i+1)*batch_size]
		seq_rep, _, _ = onehot_encoding(seq_batch, max_len, word2idx)

		X_seq = torch.LongTensor(seq_rep).cuda()


		AMP_pred_batch = AMP_model(X_seq).cpu().detach().numpy()
		AMP_pred_batch = 10**(6-AMP_pred_batch) #transform back to MICs

		if i == 0:
			AMP_pred = AMP_pred_batch
		else:
			AMP_pred = np.vstack([AMP_pred, AMP_pred_batch])

	if ensemble_id == 0:
		AMP_sum = AMP_pred
	else:
		AMP_sum += AMP_pred
	ensemble_counter += 1

AMP_pred = AMP_sum / float(ensemble_counter)
print('time taken', time.time()-t)
df_vep_mic = pd.DataFrame(data=AMP_pred, columns=col, index=seq_list)
df_vep_mic.median(axis=1)

 The input list is: ['KLLKIGLKSFARVLKKVL' 'WLGSALKIGAKLL' 'KLWNSKLARKIRTKGLKYVKNFAK'
 'LKLKSILGKLGVIL' 'RRVKRFKKFFMKLKKSVKKRVMKFFK' 'GKWLISSLVAKHL'
 'KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK' 'KLKKLRKWIYRIV' 'FLKKIWRSKLVKRL'
 'KRRRASPLWKRRRFLSMLKARAK' 'LTKWLGKLGVIL'
 'RKFKWGKLFSTAKKLYKKGKKLSKNKNFKKALK' 'KFLARLVFRKFILL'
 'KNKRFIRNLRSNLYQKIIKSTKSLL' 'KWLGKLGVILSHL' 'RKFKWGKLFSTAKKLYKKGKKLSK'
 'FIKKLWRSKLAKKLRAKGRELLK' 'RRVKRFKKFFMKL' 'VNSFKIGGFIKKLWRSKLAKKLRAK'
 'RFGSFLKKVWKSKLAKKL' 'RRVKRFKKFFRKLKKSVKKRAKEFFK' 'RHRIVRTYIAKFGLK'
 'KRKGYLRLVPEERIWQKGLWWLRRLETDSDKLQK' 'LLHFSIWRSTVLRK'
 'RHRIVRTYIAKFGLKLNEFFQENENAWYFIRNIRKRVWEVKK' 'RRVKRFKKFFKKL'
 'QPRRVKRFKKFFKKLKNSVKKRAKKF' 'KRLRAKMLNSKFIKLIKR'
 'RRASPLWKRRRFLSMLKARAKRTGYK' 'PLWKRRRFLSMLKARAKR' 'LRAKMLNSKFIKL'
 'KLHGLLTRRSLKNFWKRNLYLR' 'KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR'
 'RLRAKMRNSKLFKLTKR' 'RQEYPTKRLRAKMLNSKFIKLIKR' 'KKWRELSRLSRVLQIL'
 'RKRRRFISMLKARAKRR' 'KQKYLIKRSRAKMQNHKLFKLTKR'
 'RKFKWGSFKKILSAGKKLFKKAKKLSK' 'KWGKLFSAGKKLLKKAKKL' 'KIKWLKAMKS

,0
KLLKIGLKSFARVLKKVL,18.493744
WLGSALKIGAKLL,17.443626
KLWNSKLARKIRTKGLKYVKNFAK,46.957985
LKLKSILGKLGVIL,12.831366
RRVKRFKKFFMKLKKSVKKRVMKFFK,56.134571
GKWLISSLVAKHL,15.641453
KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK,66.547585
KLKKLRKWIYRIV,37.603188
FLKKIWRSKLVKRL,27.004911
KRRRASPLWKRRRFLSMLKARAK,56.474556


In [12]:
df_mic_55 = df_vep_mic.reset_index().rename(columns={'index': 'Sequence'})
n_gt_32=(df_mic_55.iloc[:,1:].median(axis=1).values>32).sum()
pc_gt_32=(df_mic_55.iloc[:,1:].median(axis=1).values>32).mean()*100
df_mic_55.to_csv('df_mic_55x35.csv', index=False)
print(f'{n_gt_32} of the final 55 VEPs (or {pc_gt_32} percent) have APEX-Predicted MIC > 32! Here is the final dataframe: ')
df_mic_55

42 of the final 55 VEPs (or 76.36363636363637 percent) have APEX-Predicted MIC > 32! Here is the final dataframe: 


,Sequence,E. coli ATCC11775,P. aeruginosa PAO1,P. aeruginosa PA14,S. aureus ATCC12600,E. coli AIG221,E. coli AIG222,K. pneumoniae ATCC13883,A. baumannii ATCC19606,A. muciniphila ATCC BAA-835,...,R. obeum,R. torques,S. aureus (ATCC BAA-1556) - MRSA,vancomycin-resistant E. faecalis ATCC700802,vancomycin-resistant E. faecium ATCC700221,E. coli Nissle,Salmonella enterica ATCC 9150 (BEIRES NR-515),Salmonella enterica (BEIRES NR-170),Salmonella enterica ATCC 9150 (BEIRES NR-174),L. monocytogenes ATCC 19111 (BEIRES NR-106)
0,KLLKIGLKSFARVLKKVL,7.630094,12.327852,17.703650,23.381571,10.479136,9.201456,22.415638,2.904467,6.383162,...,346.442291,143.101898,24.319881,40.748459,5.034935,514.965027,8.305558,235.542328,261.745758,15.221005
1,WLGSALKIGAKLL,12.725700,44.431915,56.736542,11.008323,11.277797,8.040901,21.578999,6.238828,30.969669,...,194.455643,47.988377,13.541011,40.069790,5.775425,354.395935,15.283323,323.942871,405.748230,20.224804
2,KLWNSKLARKIRTKGLKYVKNFAK,14.614248,9.662188,12.731038,41.280724,8.959637,11.542986,48.083843,7.042480,7.175234,...,462.761627,321.855591,53.418976,95.589005,7.445621,336.862549,4.462077,222.412766,156.180191,10.158847
3,LKLKSILGKLGVIL,9.679654,28.665686,36.115726,14.463969,10.064558,6.648721,18.667576,6.695969,8.740446,...,118.953995,40.000294,13.272173,32.564507,5.365795,245.696167,16.972363,128.598923,126.641281,24.514133
4,RRVKRFKKFFMKLKKSVKKRVMKFFK,15.252353,8.151583,13.043551,53.608166,13.172803,13.664485,92.007004,12.239097,3.642266,...,578.644897,373.900055,57.094898,95.394394,8.477136,447.937592,4.666789,153.183640,188.099930,12.393736
5,GKWLISSLVAKHL,9.333745,35.509205,44.693993,13.826393,8.518499,6.608459,16.788589,3.842312,18.526373,...,214.066330,52.649384,17.562584,47.315865,6.320087,162.949921,11.009680,124.680275,114.633347,14.756948
6,KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK,20.012367,11.588827,14.706251,44.356834,11.053272,14.438757,66.986977,11.407370,8.143622,...,499.354553,397.320770,61.039051,97.165405,7.910892,406.996185,4.791270,171.932999,167.583969,10.755083
7,KLKKLRKWIYRIV,15.005010,14.140190,23.003414,52.312584,13.605214,14.748598,69.368225,12.723369,2.595351,...,407.990234,280.034485,46.796757,90.778450,11.951981,249.537918,11.988327,77.700813,90.218796,25.637127
8,FLKKIWRSKLVKRL,10.645987,12.230603,20.745794,26.754894,15.425552,11.063894,59.602131,6.821894,5.297017,...,462.897705,222.014084,28.743210,57.977730,7.871366,250.341873,12.121559,170.172424,170.256927,24.756218
9,KRRRASPLWKRRRFLSMLKARAK,15.617238,13.151022,15.691370,51.631752,13.443766,12.150095,63.456982,8.165344,11.383955,...,540.185730,310.739410,69.627792,92.638756,8.241979,399.135193,5.169033,263.933807,201.098190,11.031645


Now, we have 2 observations:
- In the supplementary data shared there were 11 strains in the columns.
- These are the same 11 strains that re-appear in the APEX-Pathogen model repo.


In [13]:
#@title
# Source: https://gitlab.com/machine-biology-group-public/apex-pathogen

import difflib
# 1: Fetch list of 34 strains in APEX
list_strains_34=['E. coli ATCC11775', 'P. aeruginosa PAO1', 'P. aeruginosa PA14', 'S. aureus ATCC12600', 'E. coli AIG221', 'E. coli AIG222', 'K. pneumoniae ATCC13883', 'A. baumannii ATCC19606', 'A. muciniphila ATCC BAA-835', 'B. fragilis ATCC25285', 'B. vulgatus ATCC8482', 'C. aerofaciens ATCC25986', 'C. scindens ATCC35704', 'B. thetaiotaomicron ATCC29148', 'B. thetaiotaomicron Complemmented', 'B. thetaiotaomicron Mutant', 'B. uniformis ATCC8492', 'B. eggerthi ATCC27754', 'C. spiroforme ATCC29900', 'P. distasonis ATCC8503', 'P. copri DSMZ18205', 'B. ovatus ATCC8483', 'E. rectale ATCC33656', 'C. symbiosum', 'R. obeum', 'R. torques', 'S. aureus (ATCC BAA-1556) - MRSA', 'vancomycin-resistant E. faecalis ATCC700802', 'vancomycin-resistant E. faecium ATCC700221', 'E. coli Nissle', 'Salmonella enterica ATCC 9150 (BEIRES NR-515)', 'Salmonella enterica (BEIRES NR-170)', 'Salmonella enterica ATCC 9150 (BEIRES NR-174)', 'L. monocytogenes ATCC 19111 (BEIRES NR-106)']

# 2: Fetch list of 11 strains in supplementary data
list_strains_11=list(df_2.columns[1:12])

# 3: Use fuzzy string search to map the 11 chosen strains to the 34 in the APEX list to extract the mapping indices list

list_ind_apex_map=[]
for strain in list_strains_11:
  print(f'Strain being searched: {strain}')
  match_strain=difflib.get_close_matches(strain,list_strains_34,cutoff=0.9)[0]
  print(f'Strain match in list_strains_34: {match_strain}')
  ind_match=list_strains_34.index(match_strain)
  list_ind_apex_map.append(ind_match)

print('-------------')
print(list_ind_apex_map)
print('-------')
np.vstack([np.array(list_strains_11),np.array(list_strains_34)[list_ind_apex_map]]).T

Strain being searched: E. coli ATCC11775
Strain match in list_strains_34: E. coli ATCC11775
Strain being searched: P. aeruginosa PAO1
Strain match in list_strains_34: P. aeruginosa PAO1
Strain being searched: P. aeruginosa PA14
Strain match in list_strains_34: P. aeruginosa PA14
Strain being searched: S. aureus ATCC12600
Strain match in list_strains_34: S. aureus ATCC12600
Strain being searched: E. coli AIG221
Strain match in list_strains_34: E. coli AIG221
Strain being searched: E. coli AIG222
Strain match in list_strains_34: E. coli AIG222
Strain being searched: K. pneumoniae ATCC13883
Strain match in list_strains_34: K. pneumoniae ATCC13883
Strain being searched: A. baumannii ATCC19606
Strain match in list_strains_34: A. baumannii ATCC19606
Strain being searched: S. aureus (ATCC BAA-1556) - MRSA
Strain match in list_strains_34: S. aureus (ATCC BAA-1556) - MRSA
Strain being searched: vancomycin-resistant E. faecalis ATCC700802
Strain match in list_strains_34: vancomycin-resistant E. 

array([['E. coli ATCC11775', 'E. coli ATCC11775'],
       ['P. aeruginosa PAO1', 'P. aeruginosa PAO1'],
       ['P. aeruginosa PA14', 'P. aeruginosa PA14'],
       ['S. aureus ATCC12600', 'S. aureus ATCC12600'],
       ['E. coli AIG221', 'E. coli AIG221'],
       ['E. coli AIG222', 'E. coli AIG222'],
       ['K. pneumoniae ATCC13883', 'K. pneumoniae ATCC13883'],
       ['A. baumannii ATCC19606', 'A. baumannii ATCC19606'],
       ['S. aureus (ATCC BAA-1556) - MRSA',
        'S. aureus (ATCC BAA-1556) - MRSA'],
       ['vancomycin-resistant E. faecalis ATCC700802',
        'vancomycin-resistant E. faecalis ATCC700802'],
       ['vancomycin-resistant E. faecium ATCC700221',
        'vancomycin-resistant E. faecium ATCC700221']], dtype='<U45')

In [14]:
df_vep_mic.iloc[:,list_ind_apex_map].median(axis=1)

,0
KLLKIGLKSFARVLKKVL,12.327852
WLGSALKIGAKLL,12.725700
KLWNSKLARKIRTKGLKYVKNFAK,12.731038
LKLKSILGKLGVIL,13.272173
RRVKRFKKFFMKLKKSVKKRVMKFFK,13.664485
GKWLISSLVAKHL,13.826393
KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK,14.706251
KLKKLRKWIYRIV,15.005010
FLKKIWRSKLVKRL,15.425552
KRRRASPLWKRRRFLSMLKARAK,15.617238


In [15]:
#@title Issue 4b : What about the 4618 VEPs attributed to venomzone?

# Supplementary data 1:List of VEP predicted by APEX to have a median MIC ≤32 μmol<sup>-1</sup>.csv
url_2='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM4_ESM.csv'
df_venomzone_4618=pd.read_csv('https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM3_ESM.csv')
seq_vep_4618=df_venomzone_4618.Sequence.values
seq_vep_4618

array(['VNWKKVLGKIIKVAK', 'VNWKKILGKIIKVAK', 'VNWKKILGKIIKVA', ...,
       'SFKFGSFIKRMWRSKLAKKLRAK', 'KGFAKKLWNSKLARKIRTKG',
       'KFGGFLKKMWKSKLAKKLRAKGKQMLKEYANKVL'], dtype=object)

In [16]:
t=time.time()


ensemble_counter = 0
for ensemble_id in range(ensemble_num):

	AMP_model = deep_model_list[ensemble_id].cuda().eval()

	data_len = len(seq_vep_4618)
	batch_size = 3000 #change according to your GPU memory
	for i in range(int(math.ceil(data_len/float(batch_size)))):
		# if (i*batch_size) % 1000 == 0:
		# 	print ('progress', i*batch_size, data_len)

		seq_batch = seq_vep_4618[i*batch_size:(i+1)*batch_size]
		seq_rep, _, _ = onehot_encoding(seq_batch, max_len, word2idx)

		X_seq = torch.LongTensor(seq_rep).cuda()


		AMP_pred_vep_4618_batch = AMP_model(X_seq).cpu().detach().numpy()
		AMP_pred_vep_4618_batch = 10**(6-AMP_pred_vep_4618_batch) #transform back to MICs

		if i == 0:
			AMP_pred_vep_4618 = AMP_pred_vep_4618_batch
		else:
			AMP_pred_vep_4618 = np.vstack([AMP_pred_vep_4618, AMP_pred_vep_4618_batch])

	if ensemble_id == 0:
		AMP_sum_vep_4618 = AMP_pred_vep_4618
	else:
		AMP_sum_vep_4618 += AMP_pred_vep_4618
	ensemble_counter += 1

AMP_pred_vep_4618 = AMP_sum_vep_4618 / float(ensemble_counter)
print('time taken', time.time()-t)
df_mic_4618 = pd.DataFrame(data=AMP_pred_vep_4618, columns=col, index=seq_vep_4618)

time taken 14.094850063323975


In [17]:
(df_mic_4618.iloc[:,list_ind_apex_map].median(axis=1)>32).sum()

np.int64(2)

In [18]:
df_mic_vep_4618 = df_mic_4618.iloc[:,list_ind_apex_map].reset_index().rename(columns={'index': 'Sequence'})
df_mic_vep_4618['Row_Median']=df_mic_vep_4618.iloc[:,1:].median(axis=1)
df_mic_vep_4618.compare(df_venomzone_4618)

E. coli ATCC11775            P. aeruginosa PAO1             \
                  self      other               self      other   
0             2.395261   2.395565           4.305549   4.306142   
1             2.444971   2.445251           4.587688   4.588087   
2             3.865203   3.865204           8.178395   8.178099   
3             3.283432   3.283447           5.391380   5.391641   
4             4.452035   4.452406           8.562277   8.562624   
...                ...        ...                ...        ...   
4613         22.812986  22.812094          35.445610  35.444508   
4614         31.992935  31.995213          32.145164  32.144802   
4615         35.981560  35.981450          22.347075  22.348135   
4616         37.907761  37.903316          24.454056  24.450533   
4617         45.741459  45.743080          20.765965  20.763344   

     P. aeruginosa PA14            S. aureus ATCC12600             \
                   self      other                self      other   
0              6.397162   6.398497           12.660839  12.660364   
1              7.330486   7.331506           12.351071  12.349998   
2             12.738499  12.739119           13.595599  13.594274   
3              8.721972   8.722420           15.376712  15.375223   
4             12.080617  12.081948           16.056273  16.055687   
...                 ...        ...                 ...        ...   
4613          31.994577  31.992676           36.377022  36.378853   
4614          31.264322  31.267902           67.698586  67.693990   
4615          31.995762  31.999289           42.438538  42.438190   
4616          32.000313  31.999353           46.973465  46.974266   
4617          24.325506  24.323252           38.960213  38.959360   

     E. coli AIG221             ... A. baumannii ATCC19606             \
               self      other  ...                   self      other   
0          3.565358   3.565669  ...               1.953861   1.953823   
1          3.500371   3.500514  ...               1.775656   1.775475   
2          4.217658   4.217413  ...               2.534902   2.534552   
3          4.554776   4.555102  ...               2.184039   2.183827   
4          5.073564   5.073361  ...               3.367932   3.367759   
...             ...        ...  ...                    ...        ...   
4613      17.897100  17.894577  ...              11.952883  11.953753   
4614      24.378105  24.377424  ...              26.986103  26.988972   
4615      25.398956  25.397337  ...              13.682104  13.681305   
4616      18.687216  18.684095  ...              20.903606  20.904915   
4617      30.475723  30.474728  ...              23.683481  23.682247   

     S. aureus (ATCC BAA-1556) - MRSA             \
                                 self      other   
0                           12.897255  12.896708   
1                           12.998236  12.996942   
2                           14.428854  14.427275   
3                           15.797008  15.795485   
4                           16.526360  16.526554   
...                               ...        ...   
4613                        43.896065  43.896053   
4614                        70.276810  70.269554   
4615                        62.606079  62.607900   
4616                        61.688679  61.684193   
4617                        59.718792  59.718880   

     vancomycin-resistant E. faecalis ATCC700802              \
                                            self       other   
0                                      28.987675   28.987717   
1                                      30.017294   30.013561   
2                                      32.894585   32.891760   
3                                      32.786301   32.781990   
4                                      34.026867   34.025610   
...                                          ...         ...   
4613                                   64.217766   64.217020   
4614                            

# Run agentic inference on Abacus and compare the results!

The `ABACUS_AGENTIC_FLOW` directory in the github repo has the dataframes generated by the ABACUS agents.
The 2 files are:
- apex_results_58veps.csv
- apex_results_4618veps.csv

In [19]:
df_abacus_55=pd.read_csv('https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/refs/heads/main/ABACUS_AGENTIC_FLOW/apex_results_58veps.csv')
df_abacus_55.drop_duplicates(subset='sequence', inplace=True)
df_abacus_55.reset_index(drop=True, inplace=True)
df_abacus_55.shape, df_mic_55.shape

((55, 35), (55, 35))

In [20]:
df_abacus_55.columns=df_mic_55.columns
df_abacus_55.compare(df_mic_55)

E. coli ATCC11775            P. aeruginosa PAO1             \
                self      other               self      other   
0           7.630084   7.630094          12.327843  12.327852   
1          12.725692  12.725700          44.431915  44.431915   
2          14.614241  14.614248           9.662188   9.662188   
3           9.679655   9.679654          28.665695  28.665686   
4          15.252355  15.252353           8.151580   8.151583   
5           9.333747   9.333745          35.509200  35.509205   
6          20.012377  20.012367          11.588822  11.588827   
7          15.005005  15.005010          14.140175  14.140190   
8          10.645979  10.645987          12.230598  12.230603   
9          15.617233  15.617238          13.151019  13.151022   
10         15.738556  15.738573          44.892937  44.892952   
11         25.597588  25.597588          13.594069  13.594073   
12          8.443631   8.443628          39.047855  39.047852   
13         17.200840  17.200848          14.331891  14.331902   
14         13.953741  13.953738          39.116260  39.116261   
15         21.671060  21.671083          15.176439  15.176443   
16         21.297642  21.297649          12.204844  12.204838   
17         16.251911  16.251913          16.033860  16.033859   
18         17.268026  17.268036          13.108653  13.108653   
19         17.360054  17.360056          16.244677  16.244680   
20         22.907892  22.907887          13.103688  13.103691   
21         19.044067  19.044075          28.459158  28.459162   
22         37.289040  37.289051          27.407639  27.407642   
23         26.701962  26.701971          66.334270  66.334290   
24         27.961014  27.961018          28.761112  28.761114   
25         24.740253  24.740255          18.675580  18.675594   
26         39.113450  39.113468          15.689786  15.689789   
27         12.948614  12.948606           9.550250   9.550248   
28         23.094725  23.094742          18.931505  18.931503   
29         18.288270  18.288267          21.666471  21.666470   
30         21.809298  21.809286          20.565320  20.565325   
31         21.079790  21.079800          22.315624  22.315628   
32         30.508427  30.508442          22.438437  22.438438   
33         26.653980  26.653982          21.296505  21.296501   
34         32.335163  32.335186          23.378819  23.378830   
35         21.413181  21.413197          26.741512  26.741528   
36         26.757350  26.757355          24.797256  24.797258   
37         39.161095  39.161110          20.119102  20.119114   
38         16.672241  16.672245          10.468809  10.468811   
39         15.742534  15.742529          12.737252  12.737244   
40          8.808943   8.808938          15.832975  15.832972   
41         22.811436  22.811430          11.372072  11.372075   
42         27.910883  27.910898          13.653086  13.653087   
43         23.793373  23.793390          14.264073  14.264081   
44         14.850611  14.850614          17.939493  17.939480   
45         18.198414  18.198421          17.718899  17.718912   
46          9.472136   9.472139          31.816296  31.816320   
47         35.489720  35.489700          19.641354  19.641361   
48         25.612055  25.612051          19.913670  19.913664   
49         19.451294  19.451292          22.066479  22.066479   
50         16.700928  16.700947          22.194447  22.194468   
51         21.223814  21.223808          15.347343  15.347348   
52         32.191322  32.191319          22.890743  22.890741   
53         23.529293  23.529299          20.085443  20.085438   
54          8.152453   8.152450          27.320475  27.320475   

   P. aeruginosa PA14            S. aureus ATCC12600             \
                 self      other                self      other   
0           17.703646  17.703650           23.381582  23.381571   
1           56.736520  56.736542           11.008322  11.008323   
2           12.731043  12.731

In [21]:
abs(df_abacus_55.iloc[:,1:]-df_mic_55.iloc[:,1:].values).max().max()

0.004

The results match!!!